In [ ]:
# установка всех необходимых библиотек (если они не установлены)
!pip install requests maxminddb tqdm pandas

In [ ]:
import requests, json, maxminddb, tqdm

addr_to_country = dict()

with maxminddb.Reader('./mmdb/GeoLite2-Country.mmdb') as reader:
    response = requests.get('https://cloud-api.yandex.net/v1/disk/public/resources', {'public_key':'https://disk.yandex.ru/d/SRpp1XyYiXiZlg', 'limit':'100'})
    for item in tqdm.tqdm(response.json()['_embedded']['items']):
        if item['name'].startswith('nodes_of'):
            continue
        json_file = requests.get(item['file'])
        json_content = json.loads(json_file.content)
        for sess in json_content:
            if sess['addr'] in addr_to_country: continue
            res = reader.get(sess['addr'])
            if res is None: continue
            addr_to_country[sess['addr']] = res['registered_country']['names']['en']

import pandas as pd

pd.DataFrame(addr_to_country.items(), columns=['addr', 'country']).to_csv('./address_to_country.csv', index=False)